# Телеграмм Бот о Правилах Баскетбола

## Структура проекта

1. ***Подготовьте данные для поиска***
* Основные зависимости
* Imports
* Сбор заголовков статей из категории
* Очистка текста (теги, пробелы, короткие секции)
* Фрагментация документов
* Токенизация и сохранение результата
2. ***Search (Поиск)***
* Поиск по базе знаний по эмбеддингам
* Поддержка поиска по синонимам/FAQ
3. ***Ask***
* Телеграм-бот
* Логика ответа: FAQ -> поиск по базе -> генерация ответа
* Логирование и отладка

# Проект

## 1. Подготовьте данные для поиска

### Основные зависимости

In [ ]:
!pip install openai mwclient mwparserfromhell tiktoken aiogram pandas wikipedia-api python-dotenv nest_asyncio

In [ ]:
# Отключим предупреждения в колабе. Будет меньше лишней информации в выводе
import warnings
warnings.filterwarnings('ignore')

### Imports

In [ ]:
import os
import re
import json
import numpy as np
import pandas as pd
import tiktoken
import nest_asyncio
import openai
from getpass import getpass
from tqdm.auto import tqdm

# aiogram
from aiogram import Bot, Dispatcher, F, Router
from aiogram.types import Message
from aiogram.enums import ParseMode
from aiogram.client.default import DefaultBotProperties

# Для работы с MediaWiki и парсинга wiki-разметки
import mwclient  # для загрузки примеров статей Википедии через API
import mwparserfromhell  # для парсинга MediaWiki разметки

### Сбор заголовков статей из категории

In [ ]:
CATEGORY_TITLE = 'Category:Rules of basketball'
WIKI_SITE = "en.wikipedia.org"
SECTIONS_TO_IGNORE = {
    "See also", "References", "External links", "Further reading",
    "Footnotes", "Bibliography", "Sources", "Citations", "Literature",
    "Notes", "Photo gallery", "Gallery", "Photos",
    "References and sources", "References and notes"
}

# Рекурсивная функция получения названий статей
def titles_from_category(category, max_depth=2):
    titles = set()
    for member in category.members():
        if isinstance(member, mwclient.page.Page):
            titles.add(member.name)
        elif isinstance(member, mwclient.listing.Category) and max_depth > 0:
            titles |= titles_from_category(member, max_depth - 1)
    return titles

# Обработка секций внутри статьи
def all_subsections_from_section(
    section: mwparserfromhell.wikicode.Wikicode,
    parent_titles: list[str],
    sections_to_ignore: set[str],
) -> list[tuple[list[str], str]]:
    headings = [str(h) for h in section.filter_headings()]
    title = headings[0]
    if title.strip("=" + " ") in sections_to_ignore:
        return []

    titles = parent_titles + [title]
    full_text = str(section)
    section_text = full_text.split(title)[1]

    if len(headings) == 1:
        return [(titles, section_text)]
    else:
        first_subtitle = headings[1]
        section_text = section_text.split(first_subtitle)[0]

        results = [(titles, section_text)]
        for subsection in section.get_sections(levels=[len(titles) + 1]):
            results.extend(all_subsections_from_section(subsection, titles, sections_to_ignore))
        return results

# Разбивка статьи на секции
def all_subsections_from_title(
    title: str,
    sections_to_ignore: set[str] = SECTIONS_TO_IGNORE,
    site_name: str = WIKI_SITE,
) -> list[tuple[list[str], str]]:
    site = mwclient.Site(site_name)
    page = site.pages[title]
    text = page.text()
    parsed_text = mwparserfromhell.parse(text)

    headings = [str(h) for h in parsed_text.filter_headings()]
    summary_text = str(parsed_text).split(headings[0])[0] if headings else str(parsed_text)

    results = [([title], summary_text)]
    for subsection in parsed_text.get_sections(levels=[2]):
        results.extend(all_subsections_from_section(subsection, [title], sections_to_ignore))
    return results

# Фильтрация секций (по длине и наличию английских символов)
def keep_section(section: tuple[list[str], str]) -> bool:
    _, text = section
    if len(text) < 50 or not re.search(r"[a-zA-Z]{3,}", text):
        return False
    return True

# --- СБОР ---
site = mwclient.Site(WIKI_SITE)
category_page = site.pages[CATEGORY_TITLE]
titles = list(set(titles_from_category(category_page, max_depth=2)))
print(f"🔍 Найдено {len(titles)} уникальных статей")

# Разбивка статей на секции
wikipedia_sections = []
for title in titles:
    try:
        wikipedia_sections.extend(all_subsections_from_title(title))
    except Exception as e:
        print(f"⚠️ Ошибка в статье {title}: {e}")

# Очистка и фильтрация
wikipedia_sections = [ws for ws in wikipedia_sections if keep_section(ws)]
print(f"✅ Отобрано {len(wikipedia_sections)} релевантных секций из {len(titles)} статей")

# Пример вывода
for i, (titles, content) in enumerate(wikipedia_sections[:5]):
    print("📄", " > ".join(titles))
    print(content.strip()[:400], "\n" + "-"*50)


🔍 Найдено 15 уникальных статей
✅ Отобрано 89 релевантных секций из 15 статей
📄 Category:Basketball penalties
[[Category:Basketball terminology|Penalties]]
[[Category:Rules of basketball|Penalties]]
[[Category:Sports penalties]] 
--------------------------------------------------
📄 Category:Scoring (basketball)
[[Category:Scoring (sport)|Basketball]]
[[Category:Rules of basketball]] 
--------------------------------------------------
📄 Elam Ending
{{short description|Basketball untimed format}}
The '''Elam Ending''', also known as '''final target score'''<ref name=":0">{{Cite web |date=2022-09-02 |title=G League to use Elam Ending format in OT games |url=https://www.espn.com/nba/story/_/id/34504428/nba-g-league-utilize-elam-ending-format-games-2022-23-season |access-date=2023-04-09 |website=ESPN.com |language=en}}</ref> or '''winning score' 
--------------------------------------------------
📄 Elam Ending > ==Format==
Instead of a game clock, teams play to a target score, with the [[sho

### Очистка текста (теги ссылок, пробелы и короткие секции)

In [ ]:
# Очистка текста секции с помощью mwparserfromhell
def clean_section(section: tuple[list[str], str]) -> tuple[list[str], str]:
    import mwparserfromhell
    import re

    titles, raw_text = section

    # Основная очистка wiki-разметки
    parsed = mwparserfromhell.parse(raw_text)
    cleaned_text = parsed.strip_code().strip()

    # Дополнительная ручная фильтрация артефактов (если strip_code не сработал)
    cleaned_text = re.sub(r"\bthumb\b", "", cleaned_text)
    cleaned_text = re.sub(r"\b\d{2,4}px\b", "", cleaned_text)  # размеры типа 170px
    cleaned_text = re.sub(r"\bright\b|\bleft\b|\bcenter\b", "", cleaned_text)
    cleaned_text = re.sub(r"\|+", "", cleaned_text)  # одиночные/двойные |
    cleaned_text = re.sub(r"\[\]", "", cleaned_text)

    # Очистка пробелов и строк
    cleaned_text = re.sub(r"\n{2,}", "\n", cleaned_text)
    cleaned_text = re.sub(r"[ \t]{2,}", " ", cleaned_text)
    cleaned_text = cleaned_text.strip()

    # Очистка заголовков
    clean_titles = [t.strip("= ").strip() for t in titles if t.strip()]

    return (clean_titles, cleaned_text)

# Применим очистку ко всем секциям
wikipedia_sections = [clean_section(ws) for ws in wikipedia_sections]

# Фильтрация: оставляем только информативные секции (от 5 слов и более)
def keep_section(section: tuple[list[str], str]) -> bool:
    _, text = section
    return len(text.split()) >= 5

original_num_sections = len(wikipedia_sections)
wikipedia_sections = [ws for ws in wikipedia_sections if keep_section(ws)]
print(f"Отфильтровано {original_num_sections - len(wikipedia_sections)} секций, осталось {len(wikipedia_sections)} секций.")

# Визуализация первых 10
for ws in wikipedia_sections[:10]:
    print(ws[0])
    display(ws[1][:50] + "...")
    print()

Отфильтровано 4 секций, осталось 85 секций.
['Elam Ending']


'The Elam Ending, also known as final target score ...'


['Elam Ending', 'Format']


'Instead of a game clock, teams play to a target sc...'


['Elam Ending', 'History', 'The Basketball Tournament']


...

'Typewritten first draft of the rules of basketball...'


['Rules of basketball', 'Players, substitutes, teams and teammates']


"Naismith's original rules did not specify how many..."


['Rules of basketball', 'Shot clock and time limits']


'The first time restriction on possession of the ba...'

### Фрагментация документов

In [ ]:
TOKENIZER_MODEL = "text-embedding-3-small"

# Функция подсчета токенов
def num_tokens(text: str, model: str = TOKENIZER_MODEL) -> int:
    """Возвращает число токенов в строке."""
    encoding = tiktoken.encoding_for_model(model)
    return len(encoding.encode(text))

# Функция разделения строк
def halved_by_delimiter(string: str, delimiter: str = "\n") -> list[str, str]:
    """Разделяет строку надвое с помощью разделителя (delimiter), пытаясь сбалансировать токены с каждой стороны."""

    # Делим строку на части по разделителю, по умолчанию \n - перенос строки
    chunks = string.split(delimiter)
    if len(chunks) == 1:
        return [string, ""]  # разделитель не найден
    elif len(chunks) == 2:
        return chunks  # нет необходимости искать промежуточную точку
    else:
        # Считаем токены
        total_tokens = num_tokens(string)
        halfway = total_tokens // 2
        # Предварительное разделение по середине числа токенов
        best_diff = halfway
        # В цикле ищем какой из разделителей, будет ближе всего к best_diff
        for i, chunk in enumerate(chunks):
            left = delimiter.join(chunks[: i + 1])
            left_tokens = num_tokens(left)
            diff = abs(halfway - left_tokens)
            if diff >= best_diff:
                break
            else:
                best_diff = diff
        left = delimiter.join(chunks[:i])
        right = delimiter.join(chunks[i:])
        # Возвращаем левую и правую часть оптимально разделенной строки
        return [left, right]


# Функция обрезает строку до максимально разрешенного числа токенов
def truncated_string(
    string: str, # строка
    model: str, # модель
    max_tokens: int, # максимальное число разрешенных токенов
    print_warning: bool = True, # флаг вывода предупреждения
) -> str:
    """Обрезка строки до максимально разрешенного числа токенов."""
    encoding = tiktoken.encoding_for_model(model)
    encoded_string = encoding.encode(string)
    # Обрезаем строку и декодируем обратно
    truncated_string = encoding.decode(encoded_string[:max_tokens])
    if print_warning and len(encoded_string) > max_tokens:
        print(f"Предупреждение: Строка обрезана с {len(encoded_string)} токенов до {max_tokens} токенов.")
    # Усеченная строка
    return truncated_string

# Функция делит секции статьи на части по максимальному числу токенов
def split_strings_from_subsection(
    subsection: tuple[list[str], str],  # (заголовки, текст)
    max_tokens: int = 1000,
    model: str = TOKENIZER_MODEL,
    max_recursion: int = 5,
    min_tokens: int = 30
) -> list[dict]:
    """
    Делит текст секции на фрагменты (dict с полями: titles, text, tokens).
    Использует рекурсивное деление по разделителям до достижения max_tokens.
    """
    titles, text = subsection
    string = "\n\n".join(titles + [text])
    num_tokens_in_string = num_tokens(string, model)

    # 1. Если строка в пределах лимита и не короткая
    if num_tokens_in_string <= max_tokens:
        if num_tokens_in_string < min_tokens:
            return []
        return [{
            "titles": titles,
            "text": text.strip(),
            "tokens": num_tokens_in_string
        }]

    # 2. Если достигнут предел рекурсии — обрезаем
    if max_recursion == 0:
        truncated = truncated_string(string, model=model, max_tokens=max_tokens)
        token_count = num_tokens(truncated, model)
        if token_count < min_tokens:
            return []
        return [{
            "titles": titles,
            "text": truncated.strip(),
            "tokens": token_count
        }]

    # 3. Пробуем разделить по \n\n, \n, . (сначала более логичные границы)
    for delimiter in ["\n\n", "\n", ". "]:
        left, right = halved_by_delimiter(text, delimiter)
        if not left.strip() or not right.strip():
            continue  # пустая сторона — пробуем следующий разделитель

        results = []
        for half in [left, right]:
            half_subsection = (titles, half)
            half_results = split_strings_from_subsection(
                half_subsection,
                max_tokens=max_tokens,
                model=model,
                max_recursion=max_recursion - 1,
                min_tokens=min_tokens
            )
            results.extend(half_results)
        return results

    # 4. Если вообще ничего не помогло — усечём
    truncated = truncated_string(string, model=model, max_tokens=max_tokens)
    token_count = num_tokens(truncated, model)
    if token_count < min_tokens:
        return []
    return [{
        "titles": titles,
        "text": truncated.strip(),
        "tokens": token_count
    }]

# Делим секции на части
MAX_TOKENS = 1600
wikipedia_strings = []
for section in wikipedia_sections:
    wikipedia_strings.extend(split_strings_from_subsection(section, max_tokens=MAX_TOKENS))

print(f"{len(wikipedia_sections)} секций Википедии поделены на {len(wikipedia_strings)} строк.")

# Напечатаем пример строки
print(wikipedia_strings[1])

85 секций Википедии поделены на 86 строк.
{'titles': ['Elam Ending', 'Format'], 'text': 'Instead of a game clock, teams play to a target score, with the shot clock still enforced. The first team to meet or exceed the target score wins, so there is no overtime. The winning score can be a walk-off field goal (two-point or three-point) or a free throw. This format has been compared to how streetball is typically played, as street basketball games are typically played to a target score, e.g. 21 or 15.\nNick Elam devised this system because he was frustrated with stalling and passive play by a leading team and intentionally fouling by a losing team. Elam proposed that his solution, which turns off the game clock, addresses these issues.', 'tokens': 147}


### Токенизация и сохранение результата

In [ ]:
EMBEDDING_MODEL = "text-embedding-3-small"

# Ввод ключа
os.environ["OPENAI_API_KEY"] = getpass("Введите OpenAI API Key:")
openai.api_key = os.environ["OPENAI_API_KEY"]

# Получение эмбеддинга
def get_embedding(text: str, model: str = EMBEDDING_MODEL) -> list[float]:
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

# Генерация базы знаний
kb = []
for item in tqdm(wikipedia_strings):
    question = " / ".join(item["titles"])  # заголовки
    answer = item["text"]                  # очищенный текст
    tokens = len(tiktoken.encoding_for_model(EMBEDDING_MODEL).encode(answer))
    embedding = get_embedding(answer)     # эмбеддинг по answer

    kb.append({
        "question": question,
        "answer": answer,
        "embedding": embedding,
        "tokens": tokens
    })

# Сохраняем в JSON
SAVE_PATH = "kb_base.json"
with open(SAVE_PATH, "w", encoding="utf-8") as f:
    json.dump(kb, f, ensure_ascii=False, indent=2)

print(f"✅ Сохранено {len(kb)} записей в файл '{SAVE_PATH}'")


Введите OpenAI API Key:··········


  0%|          | 0/86 [00:00<?, ?it/s]

✅ Сохранено 86 записей в файл 'kb_base.json'


## 2. Search (Поиск)

In [ ]:
# Настройки
EMBEDDING_MODEL = "text-embedding-3-small"
MIN_SCORE_THRESHOLD = 0.5
TOP_N = 7
KB_PATH = "kb_base.json"

# Ввод ключа и инициализация клиента
os.environ["OPENAI_API_KEY"] = getpass("Введите OpenAI API Key:")
client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def translate_to_english(text: str) -> str:
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": (
                "You are a translator that translates questions related to basketball rules "
                "into English using correct terminology. Use basketball terms like "
                "'traveling violation', 'personal foul', etc. Only return the translation."
            )},
            {"role": "user", "content": text}
        ],
        temperature=0
    )
    return response.choices[0].message.content.strip()

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def search_kb(question: str, kb_path: str = KB_PATH, top_n: int = TOP_N) -> list[str]:
    lowered_question = question.lower()
    with open(kb_path, "r", encoding="utf-8") as f:
        kb = json.load(f)

    # Поиск точного совпадения вопроса или синонима
    for item in kb:
        if lowered_question == item["question"].strip().lower():
            return [item["answer"]]
        for syn in item.get("synonyms", []):
            if syn.lower() in lowered_question or lowered_question in syn.lower():
                return [item["answer"]]

    # Семантический поиск по эмбеддингу
    translated = translate_to_english(question)
    response = client.embeddings.create(input=[translated], model=EMBEDDING_MODEL)
    query_embedding = np.array(response.data[0].embedding)
    scored_results = [
        (item["answer"], score)
        for item in kb
        if "embedding" in item and (score := cosine_similarity(query_embedding, np.array(item["embedding"]))) >= MIN_SCORE_THRESHOLD
    ]
    scored_results.sort(key=lambda x: x[1], reverse=True)
    top_answers = [x[0] for x in scored_results[:top_n]]

    # Если совсем ничего — пусто
    return top_answers

def generate_final_answer(question: str, context_list: list[str]) -> str:
    if not context_list:
        return "К сожалению, я не нашёл подходящей информации в базе знаний"

    context = "\n\n".join(context_list)
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": (
                "Ты помощник, объясняющий правила баскетбола простым языком. "
                "На основе представленной информации, дай связный и точный ответ на русском языке."
            )},
            {"role": "user", "content": f"Вот информация:\n\n{context}\n\nОтветь на вопрос: {question}"}
        ],
        temperature=0.7
    )
    return response.choices[0].message.content.strip()

# Пример вызова
question = "Какие бывают фолы в баскетболе?"
top_contexts = search_kb(question)
final_answer = generate_final_answer(question, top_contexts)

print("🔍 Ответ:")
print(final_answer)

Введите OpenAI API Key:··········
🔍 Ответ:
В баскетболе существуют различные виды фолов. Вот основные из них:

1. Обычный фол (Personal Foul): это нарушение, совершенное игроком в отношении противника, например, физический контакт или препятствие движению. При этом команда, в которую совершен фол, получает преимущество в виде штрафных бросков или владения мячом.

2. Технический фол (Technical Foul): это нарушение, совершенное командой или ее членом вне игровой ситуации, например, неправильное поведение, ругань или излишняя реакция на судейство. За технический фол противоположная команда получает один штрафной бросок и владение мячом.

3. Флагрантный фол (Flagrant Foul): это наиболее серьезное нарушение, которое может привести к травме игрока или явному намерению повредить соперника. За флагрантный фол команде, пострадавшей от нарушения, начисляются штрафные броски и она получает владение мячом.

4. Неспортивный фол (Unsportsmanlike Foul): это фол, который свидетельствует о неспортивном

## 3. Ask

In [ ]:
# FAQ с синонимами
NEW_FAQ = [
    # ПРОБЕЖКА
    {
        "question": "Что такое пробежка?",
        "answer": "Пробежка — это нарушение, когда игрок делает более двух шагов с мячом без ведения.",
        "synonyms": ["пробежка", "traveling", "traveling violation", "нарушение шагов", "шаги без ведения", "walk violation", "walk", "перенос"]
    },
    # ДВОЙНОЕ ВЕДЕНИЕ
    {
        "question": "Что такое двойное ведение?",
        "answer": "Двойное ведение — нарушение, когда игрок начал ведение, остановился, а затем снова начал вести. Мяч переходит сопернику.",
        "synonyms": ["двойное ведение", "double dribble", "double dribbling", "нарушение двойное ведение"]
    },
    # МАТЧ И ЧЕТВЕРТИ
    {
        "question": "Сколько длится матч?",
        "answer": "В НБА матч длится 48 минут (4 четверти по 12 минут), в FIBA — 40 минут (4 четверти по 10 минут).",
        "synonyms": ["длительность матча", "сколько идёт матч", "продолжительность игры", "время игры", "how long is a game", "четверти"]
    },
    {
        "question": "Сколько длится четверть?",
        "answer": "В НБА четверть — 12 минут, в FIBA — 10 минут.",
        "synonyms": ["длительность четверти", "четверть сколько минут", "сколько минут четверть", "длится четверть", "сколько минут в четверти", "четверть длительность"]
    },
    {
        "question": "А матч длиннее?",
        "answer": "В НБА матч длиннее, чем в FIBA — разница 8 минут.",
        "synonyms": ["матч длиннее", "match longer", "nba матч длиннее", "fiba матч длиннее"]
    },
    # КОМАНДА/ИГРОКИ
    {
        "question": "Сколько игроков в команде?",
        "answer": "На площадке одновременно по 5 игроков от каждой команды.",
        "synonyms": ["сколько человек в команде", "игроков на площадке", "игроков на поле", "team size", "сколько играет"]
    },
    {
        "question": "Можно ли играть втроём?",
        "answer": "Классический баскетбол — 5×5, но есть разновидность 3×3 (стритбол, FIBA 3x3).",
        "synonyms": ["3 на 3", "трое на трое", "троём", "трое", "играть втроём", "стритбол"]
    },
    # ФОЛЫ
    {
        "question": "Что такое фол?",
        "answer": "Фол — это нарушение, связанное с недопустимым физическим контактом или неспортивным поведением.",
        "synonyms": ["фол", "нарушение", "personal foul", "что такое фол", "foul", "нарушение правил", "контакт"]
    },
    {
        "question": "Какие бывают фолы?",
        "answer": "В баскетболе есть персональные, технические, неспортивные и дисквалифицирующие фолы.",
        "synonyms": ["виды фолов", "типы фолов", "фолы в баскетболе", "категории фолов", "виды нарушений"]
    },
    {
        "question": "Что такое технический фол?",
        "answer": "Технический фол — это нарушение дисциплины (споры с судьёй, неспортивное поведение и т.д.).",
        "synonyms": ["технический фол", "technical foul", "нарушение дисциплины", "технарь", "тех"]
    },
    {
        "question": "Что такое неспортивный фол?",
        "answer": "Неспортивный фол — это грубое нарушение (опасная игра, умышленный фол).",
        "synonyms": ["неспортивный фол", "unsportsmanlike foul", "умышленный фол", "грубый фол"]
    },
    {
        "question": "Что такое дисквалифицирующий фол?",
        "answer": "Дисквалифицирующий фол — нарушение, после которого игрок удаляется из матча.",
        "synonyms": ["дисквалифицирующий фол", "disqualifying foul", "удаление", "фол с удалением"]
    },
    {
        "question": "Сколько фолов можно получить?",
        "answer": "Игрок удаляется после 5 фолов (FIBA) или 6 фолов (NBA) за матч.",
        "synonyms": ["лимит фолов", "максимум фолов", "фолы игрока", "ограничение фолов", "персональные фолы"]
    },
    {
        "question": "Что такое командный фол?",
        "answer": "Командный фол — это общее количество фолов команды за четверть. После 4 фолов соперник получает право на штрафные.",
        "synonyms": ["командные фолы", "командный фол", "total team fouls", "наказание за фолы"]
    },
    {
        "question": "Можно ли фолить специально?",
        "answer": "Фолы специально разрешены, но могут привести к штрафным броскам соперника.",
        "synonyms": ["фолить специально", "умышленный фол", "намеренный фол"]
    },
    # МЯЧ, КОРЗИНА, ПЛОЩАДКА
    {
        "question": "Размер баскетбольного мяча?",
        "answer": "Мужской — №7 (около 24 см, 600–650 г), женский — №6.",
        "synonyms": ["размер мяча", "вес мяча", "стандартный мяч", "basketball size", "диаметр мяча"]
    },
    {
        "question": "Высота кольца?",
        "answer": "Высота кольца — 3,05 м от пола.",
        "synonyms": ["высота баскетбольного кольца", "кольцо высота", "basket height", "кольцо сколько метров"]
    },
    {
        "question": "Размер площадки?",
        "answer": "Стандартная площадка: 28×15 метров (FIBA) или 28,65×15,24 м (NBA).",
        "synonyms": ["размер площадки", "размер поля", "basketball court size", "размеры баскетбольной площадки"]
    },
    {
        "question": "Что такое трехсекундная зона?",
        "answer": "Это зона под кольцом, где игрокам нападения нельзя находиться более 3 секунд подряд.",
        "synonyms": ["три секунды", "трёхсекундная зона", "three-second rule", "зона под кольцом"]
    },
    {
        "question": "Что такое штрафная линия?",
        "answer": "Штрафная линия — это линия, с которой выполняются штрафные броски, расстояние 4,57 м от щита.",
        "synonyms": ["штрафная линия", "free throw line", "penalty line"]
    },
    # СЧЁТ / ОЧКИ / ШТРАФНЫЕ / БРОСКИ
    {
        "question": "За что дают 3 очка?",
        "answer": "3 очка — за бросок в корзину с дистанции за трехочковой линией.",
        "synonyms": ["3 очка", "три очка", "трёхочковый", "трёхочковый бросок", "трешка"]
    },
    {
        "question": "Сколько очков за бросок?",
        "answer": "Обычно: 1 очко — штрафной бросок, 2 очка — обычный бросок, 3 очка — из-за дуги.",
        "synonyms": ["сколько очков за бросок", "очков за бросок", "points for shot", "очки за попадание"]
    },
    {
        "question": "Что такое штрафной бросок?",
        "answer": "Это попытка набрать 1 очко, выполняется с линии штрафных после нарушения.",
        "synonyms": ["штрафной", "штрафные броски", "free throw", "penalty shot"]
    },
    {
        "question": "Что такое дабл-дабл?",
        "answer": "Дабл-дабл — двузначные показатели по двум статистическим категориям (например, очки и подборы) в одном матче.",
        "synonyms": ["double-double", "дабл-дабл", "двойной дабл"]
    },
    {
        "question": "Что такое трипл-дабл?",
        "answer": "Трипл-дабл — двузначные показатели по трём категориям (например, очки, подборы, передачи) за матч.",
        "synonyms": ["triple-double", "трипл-дабл", "тройной дабл"]
    },
    {
        "question": "Что такое подбор?",
        "answer": "Подбор — овладение мячом после неудачного броска по корзине.",
        "synonyms": ["подбор", "подбор мяча", "rebounds", "отскок"]
    },
    {
        "question": "Что такое блок-шот?",
        "answer": "Блок-шот — попытка помешать сопернику забросить мяч, выбив его в момент броска.",
        "synonyms": ["блок-шот", "block shot", "block", "блокировка"]
    },
    {
        "question": "Что такое ассист?",
        "answer": "Ассист — результативная передача, после которой партнер забивает мяч в кольцо.",
        "synonyms": ["ассист", "ассистент", "assist", "голевая передача", "пас на гол"]
    },
    {
        "question": "Можно ли забивать сверху?",
        "answer": "Да, забивать сверху (данк) можно, если правила не запрещают.",
        "synonyms": ["забить сверху", "данк", "slam dunk", "можно ли данк"]
    },
    # ПРАВИЛА И ОСТАНОВКИ
    {
        "question": "Для чего нужен тайм-аут?",
        "answer": "Для отдыха, обсуждения тактики и замены игроков.",
        "synonyms": ["тайм-аут", "перерыв", "таймаут"]
    },
    {
        "question": "Можно ли забивать из-за своей половины?",
        "answer": "Да, если мяч попал в корзину — засчитывается, обычно дают 3 очка.",
        "synonyms": ["бросок со своей половины", "long shot", "дальний бросок"]
    },
    {
        "question": "Что такое фастбрейк?",
        "answer": "Фастбрейк — быстрый переход команды от защиты к нападению.",
        "synonyms": ["фастбрейк", "быстрая атака", "fast break", "быстрый отрыв"]
    },
    {
        "question": "Что такое заслон?",
        "answer": "Заслон — приём, когда игрок закрывает путь сопернику, чтобы партнер получил преимущество.",
        "synonyms": ["заслон", "screen", "поставить заслон", "скрин"]
    },
    {
        "question": "Что такое подножка?",
        "answer": "Подножка — это запрещенный прием (нарушение), когда игрок ставит подножку сопернику.",
        "synonyms": ["подножка", "trip", "подставить ногу", "споткнуться"]
    },
    {
        "question": "Что такое владение мячом?",
        "answer": "Владение мячом — это когда команда или игрок контролирует мяч.",
        "synonyms": ["владение", "possessions", "контроль мяча"]
    },
    {
        "question": "Можно ли бить по рукам?",
        "answer": "Нет, бить по рукам — нарушение (фол), за это наказывают штрафными бросками.",
        "synonyms": ["можно ли бить по рукам", "фол по рукам", "удар по рукам"]
    },
    {
        "question": "Можно ли играть ногой?",
        "answer": "Намеренное касание мяча ногой запрещено (нарушение).",
        "synonyms": ["играть ногой", "можно ли ногой", "kick ball"]
    },
    {
        "question": "Что такое владение?",
        "answer": "Владение — это когда команда контролирует мяч.",
        "synonyms": ["possessions", "владение"]
    },
    {
        "question": "Что делать если мяч попал в аут?",
        "answer": "После аута мяч вводит команда, которая не нарушила правила.",
        "synonyms": ["мяч в ауте", "аут", "выход мяча", "out of bounds"]
    },
    # ДЕТСКИЕ/НАИВНЫЕ/ПОПУЛЯРНЫЕ
    {
        "question": "Зачем нужен баскетбольный щит?",
        "answer": "Щит помогает мячу попадать в кольцо и предотвращает его улетание далеко.",
        "synonyms": ["баскетбольный щит", "щит", "backboard", "зачем щит"]
    },
    {
        "question": "Сколько можно держать мяч?",
        "answer": "Максимум 5 секунд без дриблинга или передачи, 24 секунды на атаку.",
        "synonyms": ["сколько держать мяч", "ограничение времени", "shot clock", "24 секунды", "5 секунд"]
    },
    {
        "question": "Можно ли играть одной рукой?",
        "answer": "Да, правила не запрещают играть одной рукой.",
        "synonyms": ["играть одной рукой", "можно одной рукой", "одна рука"]
    },
    {
        "question": "Что делать если мяч застрял на кольце?",
        "answer": "Если мяч застрял — проводится спорный бросок (jump ball).",
        "synonyms": ["мяч застрял", "мяч на кольце", "stuck ball", "jump ball"]
    },
    {
        "question": "Что значит ассист?",
        "answer": "Ассист — это результативная передача, после которой партнер забивает мяч.",
        "synonyms": ["ассист", "ассистент", "assist", "пас на гол"]
    },
    {
        "question": "Что такое трехочковая линия?",
        "answer": "Это дуга, за которой броски оцениваются в 3 очка.",
        "synonyms": ["трёхочковая линия", "трехочковая дуга", "3pt line", "three-point line"]
    },
    {
        "question": "Можно ли касаться сетки?",
        "answer": "Касаться сетки при броске не запрещено, но запрещено мешать попаданию мяча (goaltending).",
        "synonyms": ["касаться сетки", "можно ли трогать сетку", "сетку трогать"]
    },
    {
        "question": "Что такое капитан команды?",
        "answer": "Капитан — это игрок, представляющий свою команду перед судьями.",
        "synonyms": ["капитан", "captain", "кто такой капитан"]
    },
    {
        "question": "Почему нельзя держать мяч дольше 24 секунд?",
        "answer": "24 секунды — это лимит времени на атаку. После — владение переходит сопернику.",
        "synonyms": ["24 секунды", "время на атаку", "лимит владения"]
    }
]

# Настройки и инициализация
EMBEDDING_MODEL = "text-embedding-3-small"
MIN_SCORE_THRESHOLD = 0.5
TOP_N = 7
KB_PATH = "kb_base.json"

nest_asyncio.apply()
os.environ["OPENAI_API_KEY"] = getpass("Введите OpenAI API Key:")
os.environ["TELEGRAM_BOT_TOKEN"] = getpass("Введите Telegram Bot Token:")
client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# FAQ обновление базы знаний
def update_kb_with_faq(kb_path: str, new_faq: list, embedding_model: str, client):
    if os.path.exists(kb_path):
        with open(kb_path, "r", encoding="utf-8") as f:
            kb = json.load(f)
    else:
        kb = []

    kb_dict = {item["question"].strip().lower(): item for item in kb}

    for entry in new_faq:
        q_lower = entry["question"].strip().lower()
        if q_lower in kb_dict:
            kb_dict[q_lower]["answer"] = entry["answer"]
            old_syns = set(kb_dict[q_lower].get("synonyms", []))
            new_syns = set(entry.get("synonyms", []))
            kb_dict[q_lower]["synonyms"] = list(old_syns | new_syns)
        else:
            kb_dict[q_lower] = {
                "question": entry["question"],
                "answer": entry["answer"],
                "synonyms": entry.get("synonyms", [])
            }

    for q, item in kb_dict.items():
        if "embedding" not in item or not item["embedding"]:
            response = client.embeddings.create(input=[item[
                "question"]], model=embedding_model)
            item["embedding"] = response.data[0].embedding

    updated_kb = list(kb_dict.values())
    with open(kb_path, "w", encoding="utf-8") as f:
        json.dump(updated_kb, f, ensure_ascii=False, indent=2)
    print(f"База знаний обновлена: {len(updated_kb)} записей.")

# Запуск обновления базы при старте
update_kb_with_faq(KB_PATH, NEW_FAQ, EMBEDDING_MODEL, client)

# Сервисные функции
GENERAL_PATTERN = re.compile(r"^(спасибо|понял|ясно|ок|угу|ага|ладно|хорошо|привет|здравствуй|да|нет|не знаю|ничего|норм|👌|👍)\W*$", re.IGNORECASE)

def translate_to_english(text: str) -> str:
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": (
                "You are a translator that translates questions related to basketball rules "
                "into English using correct terminology. Use basketball terms like "
                "'traveling violation', 'personal foul', etc. Only return the translation."
            )},
            {"role": "user", "content": text}
        ],
        temperature=0
    )
    return response.choices[0].message.content.strip()

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def search_kb(question: str, kb_path: str = KB_PATH, top_n: int = TOP_N) -> list[str]:
    lowered_question = question.lower()
    with open(kb_path, "r", encoding="utf-8") as f:
        kb = json.load(f)

    # Точное совпадение вопроса или синонима
    for item in kb:
        if lowered_question == item["question"].strip().lower():
            return [item["answer"]]
        for syn in item.get("synonyms", []):
            if syn.lower() in lowered_question or lowered_question in syn.lower():
                return [item["answer"]]

    # Поиск по эмбеддингу
    translated = translate_to_english(question)
    response = client.embeddings.create(input=[translated], model=EMBEDDING_MODEL)
    query_embedding = np.array(response.data[0].embedding)
    scored = [
        (item["answer"], score)
        for item in kb
        if "embedding" in item and (score := cosine_similarity(
            query_embedding, np.array(item["embedding"]))) >= MIN_SCORE_THRESHOLD
    ]
    scored.sort(key=lambda x: x[1], reverse=True)
    return [x[0] for x in scored[:top_n]]

def generate_final_answer(question: str, context_list: list[str]) -> str:
    if not context_list:
        return "К сожалению, я не нашёл подходящей информации в базе знаний"
    context = "\n\n".join(context_list)
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": (
                "Ты помощник, объясняющий правила баскетбола простым языком. "
                "На основе представленной информации, дай связный и точный ответ на русском языке."
            )},
            {"role": "user", "content": f"Вот информация:\n\n{context}\n\nОтветь на вопрос: {question}"}
        ],
        temperature=0.7
    )
    return response.choices[0].message.content.strip()

# Бот на aiogram
router = Router()

@router.message(F.text == "/start")
async def start_handler(message: Message):
    await message.answer(
        "Привет! 👋 Я — бот-помощник по правилам баскетбола.\n"
        "Задай мне вопрос, например:\n\n"
        "• Что такое пробежка?\n"
        "• Какие бывают фолы?\n"
        "• Сколько длится матч?\n\n"
        "И я постараюсь дать ответ)"
    )

@router.message(F.text == "/help")
async def help_handler(message: Message):
    try:
        with open(KB_PATH, "r", encoding="utf-8") as f:
            kb = json.load(f)
        sample = kb[0]["question"]
        await message.answer(
            f"<b>База знаний</b>\n\n"
            f"Тематика: <i>Правила баскетбола</i>\n"
            f"Записей: <b>{len(kb)}</b>\n"
            f"Пример: <i>{sample}</i>"
        )
    except Exception as e:
        await message.answer(f"Ошибка: {e}")

@router.message(F.text)
async def ask_handler(message: Message):
    text = message.text.strip().lower()
    if GENERAL_PATTERN.match(text):
        await message.answer("Обращайся, если появятся вопросы по правилам баскетбола!")
        return
    await message.answer("Думаю над ответом...")
    try:
        top_context = search_kb(text)
        reply = generate_final_answer(text, top_context)
        await message.answer(reply)
    except Exception as e:
        await message.answer(f"Ошибка: {e}")

# Запуск бота
async def start_bot():
    bot = Bot(
        token=os.environ["TELEGRAM_BOT_TOKEN"],
        default=DefaultBotProperties(parse_mode=ParseMode.HTML)
    )
    dp = Dispatcher()
    dp.include_router(router)
    print("Бот запущен.")
    await dp.start_polling(bot)

import asyncio
await start_bot()


Введите OpenAI API Key:··········
Введите Telegram Bot Token:··········
База знаний обновлена: 132 записей.
Бот запущен.
